# 05 — Emotion Detection

7-class facial emotion recognition using FER2013 dataset.

**Dataset**: Download from Kaggle → `fer2013.csv`  
Set `FER_CSV` path below.

Emotions: angry, disgust, fear, happy, sad, surprise, neutral

In [ ]:
import os
from pathlib import Path

# Use GPU card #2 (index 1). Change to '0' to use the first card.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# ⚠️  Set this to your fer2013.csv path
FER_CSV     = Path("../data/fer2013.csv")
WEIGHTS_DIR = Path("../weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMOTIONS   = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
IMG_SIZE   = 112
BATCH_SIZE = 64
EPOCHS     = 15
LR         = 1e-3

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

print(f"Device: {DEVICE}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES')}")

## Dataset — FER2013

In [ ]:
class FER2013Dataset(Dataset):
    """FER2013 CSV dataset. Columns: emotion, pixels, Usage."""

    def __init__(self, csv_path, split='Training', transform=None):
        df = pd.read_csv(csv_path)
        df = df[df['Usage'] == split].reset_index(drop=True)
        self.labels    = df['emotion'].values
        self.pixels    = df['pixels'].values
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        pixel_vals = np.array(self.pixels[idx].split(), dtype=np.uint8).reshape(48, 48)
        # Convert grayscale to RGB PIL image
        img = Image.fromarray(pixel_vals, mode='L').convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = FER2013Dataset(FER_CSV, split='Training',  transform=train_tf)
val_ds   = FER2013Dataset(FER_CSV, split='PublicTest', transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")

# Class distribution
counts = np.bincount(train_ds.labels)
plt.bar(EMOTIONS, counts, color='steelblue')
plt.title("Training class distribution")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## Model Definition

In [ ]:
class EmotionNet(nn.Module):
    EMOTIONS = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']

    def __init__(self, num_classes=7):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        in_features = base.fc.in_features  # 512
        base.fc = nn.Linear(in_features, num_classes)
        self.net = base

    def forward(self, x):
        return self.net(x)


## Training

In [ ]:
# Class weights to handle imbalance
counts    = np.bincount(train_ds.labels)
weights   = 1.0 / counts
weights   = torch.tensor(weights / weights.sum() * len(counts), dtype=torch.float32).to(DEVICE)

model     = EmotionNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(weight=weights)

history = {"train_loss": [], "train_acc": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in tqdm(train_loader, leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(imgs)
        correct    += (out.argmax(1) == labels).sum().item()
        total      += len(imgs)

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            val_correct += (model(imgs).argmax(1) == labels).sum().item()
            val_total   += len(imgs)

    tr_loss = total_loss / total
    tr_acc  = correct / total
    val_acc = val_correct / val_total
    history["train_loss"].append(tr_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(val_acc)
    scheduler.step()
    print(f"Epoch {epoch:02d}/{EPOCHS} | loss {tr_loss:.4f} | train acc {tr_acc:.3f} | val acc {val_acc:.3f}")

torch.save(model.state_dict(), WEIGHTS_DIR / "emotion_detection.pth")
print("Saved emotion_detection.pth")


## Evaluation

In [ ]:
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        all_preds.extend(model(imgs).argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=EMOTIONS))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=EMOTIONS, yticklabels=EMOTIONS)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Emotion Detection Confusion Matrix")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
